# Classify_04 — Statistical Analysis of Headline Models

**Purpose.** Perform the final *paired* statistical comparisons between the headline models
using the **five repeated user-grouped holdout runs** (seeds 42–46) that were already produced
by `Classify_01_DirectMulticlass`, `Classify_02_TwoStage`, `Classify_02b_TwoStageRegression`
and `Classify_03_DeepLearning`.

Nothing is retrained here. We load the *per-run* Macro-F1 values already saved to disk,
treat the five matched runs as paired observations, and run the pre-specified inferential and
sensitivity tests.

The five runs are matched by **seed** (which fixes the user-grouped train/holdout split), so all
models are aligned on the same seed rather than on row order.

> **Small-sample caveat.** With only *n = 5* repeated runs, inferential power is low and
> distributional assumptions cannot be verified. The paired *t*-test is the primary procedure;
> exact sign-flip permutation and Wilcoxon signed-rank are reported only as sensitivity checks.

In [1]:
# --- Imports & configuration ---
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)

# Paths (run from the project root)
DATA_DIR = Path("KKBoxData")
OUT_DIR = Path("outputs") / "statistics"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# The five repeated user-grouped holdout runs share these seeds across all models.
RUN_SEEDS = [42, 43, 44, 45, 46]
N_RUNS = len(RUN_SEEDS)

METRIC = "macro_f1"     # headline metric for every model
TOL = 1e-4             # tolerance for reproducing published summary statistics
RNG_SEED = 20240804    # fixed seed for the optional paired bootstrap

print("Project root :", Path(".").resolve())
print("Data dir     :", DATA_DIR.resolve())
print("Output dir   :", OUT_DIR.resolve())
print("Run seeds    :", RUN_SEEDS, f"(n = {N_RUNS})")

Project root : <repo>/Beyond-Binary-Churn
Data dir     : <repo>/Beyond-Binary-Churn/KKBoxData
Output dir   : <repo>/Beyond-Binary-Churn/outputs/statistics
Run seeds    : [42, 43, 44, 45, 46] (n = 5)


## 1 — Locate and load the existing per-run results

We reuse the CSVs already written by the four modelling notebooks and never retype values from
the dissertation tables. Each model's per-run Macro-F1 is looked up by its `model` label inside
the relevant `*_runs.csv`, keyed by `seed`. We confirm every model has exactly the same five run
seeds and fail loudly otherwise.

Result families and their source files:

| Key | Model | Per-run file | `model` value |
|-----|-------|--------------|---------------|
| `dl_ensemble` | Tuned deep-learning ensemble | `clf_dl_holdout_runs.csv` | `DL ensemble x3 (tuned)` |
| `direct_xgb` | Direct XGBoost | `clf_direct_holdout_runs.csv` | `XGBoost` |
| `direct_lgbm` | Direct LightGBM | `clf_direct_holdout_runs.csv` | `LightGBM` |
| `twostage_tuned` | Tuned two-stage classifier | `clf_twostage_runs.csv` | `Two-stage (tuned)` |
| `twostage_argmax` | Two-stage arg-max classifier | `clf_twostage_runs.csv` | `Two-stage (arg-max)` |
| `twostage_ratio` | Two-stage ratio regression | `clf_02b_runs.csv` | `02b regress-ratio` |
| `twostage_volume` | Two-stage volume regression | `clf_02b_runs.csv` | `02b regress-volume` |

In [2]:
# Where each model's per-run and published-summary values live.
MODEL_SPECS = {
    "dl_ensemble": dict(
        label="Tuned deep-learning ensemble",
        runs_file="clf_dl_holdout_runs.csv", model_col="model", model_value="DL ensemble x3 (tuned)",
        summary_file="clf_dl_holdout_summary.csv"),
    "direct_xgb": dict(
        label="Direct XGBoost",
        runs_file="clf_direct_holdout_runs.csv", model_col="model", model_value="XGBoost",
        summary_file="clf_direct_holdout_summary.csv"),
    "direct_lgbm": dict(
        label="Direct LightGBM",
        runs_file="clf_direct_holdout_runs.csv", model_col="model", model_value="LightGBM",
        summary_file="clf_direct_holdout_summary.csv"),
    "twostage_tuned": dict(
        label="Tuned two-stage classifier",
        runs_file="clf_twostage_runs.csv", model_col="model", model_value="Two-stage (tuned)",
        summary_file="clf_twostage_summary.csv"),
    "twostage_argmax": dict(
        label="Two-stage arg-max classifier",
        runs_file="clf_twostage_runs.csv", model_col="model", model_value="Two-stage (arg-max)",
        summary_file="clf_twostage_summary.csv"),
    "twostage_ratio": dict(
        label="Two-stage ratio regression",
        runs_file="clf_02b_runs.csv", model_col="model", model_value="02b regress-ratio",
        summary_file="clf_02b_summary.csv"),
    "twostage_volume": dict(
        label="Two-stage volume regression",
        runs_file="clf_02b_runs.csv", model_col="model", model_value="02b regress-volume",
        summary_file="clf_02b_summary.csv"),
}


def load_model_runs(key, spec):
    """Return a tidy 5-row [seed, macro_f1] frame for one model, or raise a clear error."""
    path = DATA_DIR / spec["runs_file"]
    if not path.exists():
        raise FileNotFoundError(f"[{key}] per-run file not found: {path}")
    df = pd.read_csv(path)
    for col in (spec["model_col"], "seed", METRIC):
        if col not in df.columns:
            raise ValueError(f"[{key}] column {col!r} missing in {path.name}")
    sub = df[df[spec["model_col"]] == spec["model_value"]].copy()
    if sub.empty:
        raise ValueError(f"[{key}] no rows where {spec['model_col']}=={spec['model_value']!r} "
                         f"in {path.name}")
    sub = sub[["seed", METRIC]].dropna()
    sub["seed"] = sub["seed"].astype(int)
    sub = sub.drop_duplicates(subset="seed").sort_values("seed").reset_index(drop=True)
    seeds = sorted(sub["seed"].tolist())
    if len(sub) != N_RUNS or seeds != sorted(RUN_SEEDS):
        raise ValueError(
            f"[{key}] expected {N_RUNS} runs with seeds {sorted(RUN_SEEDS)}, "
            f"but found {len(sub)} row(s) with seeds {seeds} in {path.name}")
    return sub


model_runs = {}
print(f"{'key':<18}{'label':<32}{'source file':<28}{'seeds'}")
print("-" * 100)
for key, spec in MODEL_SPECS.items():
    sub = load_model_runs(key, spec)
    model_runs[key] = sub
    print(f"{key:<18}{spec['label']:<32}{spec['runs_file']:<28}{sorted(sub['seed'].tolist())}")

print(f"\nAll {len(model_runs)} models loaded with exactly {N_RUNS} identifiable per-run values "
      f"on matching seeds.")

key               label                           source file                 seeds
----------------------------------------------------------------------------------------------------
dl_ensemble       Tuned deep-learning ensemble    clf_dl_holdout_runs.csv     [42, 43, 44, 45, 46]
direct_xgb        Direct XGBoost                  clf_direct_holdout_runs.csv [42, 43, 44, 45, 46]
direct_lgbm       Direct LightGBM                 clf_direct_holdout_runs.csv [42, 43, 44, 45, 46]
twostage_tuned    Tuned two-stage classifier      clf_twostage_runs.csv       [42, 43, 44, 45, 46]
twostage_argmax   Two-stage arg-max classifier    clf_twostage_runs.csv       [42, 43, 44, 45, 46]
twostage_ratio    Two-stage ratio regression      clf_02b_runs.csv            [42, 43, 44, 45, 46]
twostage_volume   Two-stage volume regression     clf_02b_runs.csv            [42, 43, 44, 45, 46]

All 7 models loaded with exactly 5 identifiable per-run values on matching seeds.


## 2 — Construct one paired run-level table

One row per shared seed, one column per model (`<key>_macro_f1`). Models are merged **on seed**,
so alignment is by matched run and not by row order. We assert five unique seeds and no missing
Macro-F1 values, and save the table to `outputs/statistics/headline_macro_f1_by_run.csv`.

In [3]:
# Build the paired table by merging every model on `seed`.
paired = pd.DataFrame({"seed": sorted(RUN_SEEDS)}).astype({"seed": int})
for key, sub in model_runs.items():
    col = f"{key}_macro_f1"
    paired = paired.merge(sub.rename(columns={METRIC: col}), on="seed", how="left")
paired = paired.sort_values("seed").reset_index(drop=True)

# Required headline columns must be present.
required_cols = ["seed", "dl_ensemble_macro_f1", "direct_xgb_macro_f1",
                 "direct_lgbm_macro_f1", "twostage_tuned_macro_f1"]
missing = [c for c in required_cols if c not in paired.columns]
assert not missing, f"Missing required columns: {missing}"

# Structural assertions.
assert len(paired) == N_RUNS, f"expected {N_RUNS} rows, got {len(paired)}"
assert paired["seed"].is_unique, "seeds are not unique"
f1_cols = [c for c in paired.columns if c.endswith("_macro_f1")]
assert paired[f1_cols].notna().all().all(), "missing Macro-F1 values in the paired table"

# Alignment check: each cell must equal that model's own value for the SAME seed.
for key, sub in model_runs.items():
    src = sub.set_index("seed")[METRIC]
    col = f"{key}_macro_f1"
    for s in RUN_SEEDS:
        got = paired.loc[paired["seed"] == s, col].iloc[0]
        assert np.isclose(got, src.loc[s]), f"seed alignment error for {key} at seed {s}"

out_paired = OUT_DIR / "headline_macro_f1_by_run.csv"
paired.to_csv(out_paired, index=False)
print("Saved:", out_paired)
paired

Saved: outputs/statistics/headline_macro_f1_by_run.csv


,seed,dl_ensemble_macro_f1,direct_xgb_macro_f1,direct_lgbm_macro_f1,twostage_tuned_macro_f1,twostage_argmax_macro_f1,twostage_ratio_macro_f1,twostage_volume_macro_f1
0,42,0.512229,0.436252,0.434925,0.434702,0.414809,0.345408,0.343534
1,43,0.510310,0.440650,0.438905,0.444085,0.416022,0.354118,0.351871
2,44,0.516534,0.441593,0.439849,0.434311,0.416465,0.342264,0.339230
3,45,0.514102,0.439204,0.438357,0.445846,0.413060,0.354818,0.353599
4,46,0.511024,0.438644,0.437698,0.434449,0.414602,0.345643,0.342028


## 3 — Verify the published summary statistics

For every model we recompute the mean, sample SD (ddof = 1), standard error ($SD/\sqrt{5}$),
minimum and maximum from the five per-run values, and compare the mean and SE against the values
already saved in the `*_summary.csv` files (the source of the dissertation tables). Any mismatch
larger than `TOL = 1e-4` prints a warning. The check is saved to
`outputs/statistics/headline_macro_f1_summary_check.csv`.

In [4]:
def published_summary(spec):
    """Return (dict with mean/std/se, filename) from the saved summary CSV, or (None, filename)."""
    path = DATA_DIR / spec["summary_file"]
    df = pd.read_csv(path)
    row = df[df["model"] == spec["model_value"]]
    if row.empty:
        return None, path.name
    r = row.iloc[0]
    return dict(mean=float(r["macro_f1_mean"]),
                std=float(r.get("macro_f1_std", np.nan)),
                se=float(r["macro_f1_se"])), path.name


rows = []
for key, spec in MODEL_SPECS.items():
    vals = paired[f"{key}_macro_f1"].to_numpy()
    calc_mean = float(np.mean(vals))
    calc_sd = float(np.std(vals, ddof=1))
    calc_se = calc_sd / np.sqrt(N_RUNS)
    pub, pub_file = published_summary(spec)

    if pub is None:
        mean_d = se_d = np.nan
        reproduced = False
        print(f"[warn] {key}: no published summary row found in {pub_file}")
    else:
        mean_d = abs(calc_mean - pub["mean"])
        se_d = abs(calc_se - pub["se"])
        reproduced = (mean_d <= TOL) and (se_d <= TOL)
        if mean_d > TOL:
            print(f"[warn] {key}: mean mismatch calc={calc_mean:.6f} pub={pub['mean']:.6f} "
                  f"|d|={mean_d:.2e}")
        if se_d > TOL:
            print(f"[warn] {key}: SE mismatch calc={calc_se:.6f} pub={pub['se']:.6f} "
                  f"|d|={se_d:.2e}")

    rows.append(dict(
        model=key, label=spec["label"], runs_source=spec["runs_file"], summary_source=pub_file,
        n=N_RUNS, mean=calc_mean, sd=calc_sd, se=calc_se,
        min=float(np.min(vals)), max=float(np.max(vals)),
        published_mean=(np.nan if pub is None else pub["mean"]),
        published_se=(np.nan if pub is None else pub["se"]),
        mean_abs_diff=mean_d, se_abs_diff=se_d, reproduced=reproduced))

summary_check = pd.DataFrame(rows)
ALL_REPRODUCED = bool(summary_check["reproduced"].all())
out_summary = OUT_DIR / "headline_macro_f1_summary_check.csv"
summary_check.to_csv(out_summary, index=False)
print("\nAll published summary values reproduced within tol:", ALL_REPRODUCED)
print("Saved:", out_summary)
summary_check.round(6)


All published summary values reproduced within tol: True
Saved: outputs/statistics/headline_macro_f1_summary_check.csv


,model,label,runs_source,summary_source,n,mean,sd,se,min,max,published_mean,published_se,mean_abs_diff,se_abs_diff,reproduced
0,dl_ensemble,Tuned deep-learning ensemble,clf_dl_holdout_runs.csv,clf_dl_holdout_summary.csv,5,0.512840,0.002515,0.001125,0.510310,0.516534,0.512840,0.001125,0.0,0.0,True
1,direct_xgb,Direct XGBoost,clf_direct_holdout_runs.csv,clf_direct_holdout_summary.csv,5,0.439269,0.002049,0.000917,0.436252,0.441593,0.439269,0.000917,0.0,0.0,True
2,direct_lgbm,Direct LightGBM,clf_direct_holdout_runs.csv,clf_direct_holdout_summary.csv,5,0.437947,0.001864,0.000834,0.434925,0.439849,0.437947,0.000834,0.0,0.0,True
3,twostage_tuned,Tuned two-stage classifier,clf_twostage_runs.csv,clf_twostage_summary.csv,5,0.438679,0.005775,0.002583,0.434311,0.445846,0.438679,0.002583,0.0,0.0,True
4,twostage_argmax,Two-stage arg-max classifier,clf_twostage_runs.csv,clf_twostage_summary.csv,5,0.414992,0.001337,0.000598,0.413060,0.416465,0.414992,0.000598,0.0,0.0,True
5,twostage_ratio,Two-stage ratio regression,clf_02b_runs.csv,clf_02b_summary.csv,5,0.348450,0.005659,0.002531,0.342264,0.354818,0.348450,0.002531,0.0,0.0,True
6,twostage_volume,Two-stage volume regression,clf_02b_runs.csv,clf_02b_summary.csv,5,0.346052,0.006322,0.002827,0.339230,0.353599,0.346052,0.002827,0.0,0.0,True


## 4 — Pre-specified main-model contrasts

Four contrasts, each computed as **model A minus model B** on the paired run-level Macro-F1:

1. Tuned deep-learning ensemble vs Direct XGBoost
2. Tuned deep-learning ensemble vs Tuned two-stage classifier
3. Direct XGBoost vs Tuned two-stage classifier
4. Direct XGBoost vs Direct LightGBM

For each contrast we report the mean, median, sample SD and SE of the five paired differences,
plus the win/tie/loss counts (runs where A > B, A = B, A < B).

In [5]:
# (name, column for A, column for B).  Difference = A - B.
CONTRASTS = [
    ("Tuned DL ensemble vs Direct XGBoost",  "dl_ensemble_macro_f1", "direct_xgb_macro_f1"),
    ("Tuned DL ensemble vs Tuned two-stage", "dl_ensemble_macro_f1", "twostage_tuned_macro_f1"),
    ("Direct XGBoost vs Tuned two-stage",    "direct_xgb_macro_f1",  "twostage_tuned_macro_f1"),
    ("Direct XGBoost vs Direct LightGBM",    "direct_xgb_macro_f1",  "direct_lgbm_macro_f1"),
]

# `results` is a list of per-contrast dicts, progressively filled by the following cells.
results = []
for name, colA, colB in CONTRASTS:
    a = paired[colA].to_numpy()
    b = paired[colB].to_numpy()
    d = a - b
    sd = float(np.std(d, ddof=1))
    results.append(dict(
        contrast=name, colA=colA, colB=colB, diffs=d,
        mean_diff=float(np.mean(d)), median_diff=float(np.median(d)),
        sd_diff=sd, se_diff=sd / np.sqrt(N_RUNS),
        winsA=int(np.sum(d > 0)), ties=int(np.sum(d == 0)), winsB=int(np.sum(d < 0))))

print("Paired differences (A - B) per seed:")
for r in results:
    print(f"  {r['contrast']:<40} {np.round(r['diffs'], 5)}  "
          f"mean={r['mean_diff']:+.5f}  wins(A/tie/B)={r['winsA']}/{r['ties']}/{r['winsB']}")

desc = pd.DataFrame([{k: v for k, v in r.items() if k != "diffs"} for r in results])
desc[["contrast", "mean_diff", "median_diff", "sd_diff", "se_diff",
      "winsA", "ties", "winsB"]].round(5)

Paired differences (A - B) per seed:
  Tuned DL ensemble vs Direct XGBoost      [0.07598 0.06966 0.07494 0.0749  0.07238]  mean=+0.07357  wins(A/tie/B)=5/0/0
  Tuned DL ensemble vs Tuned two-stage     [0.07753 0.06622 0.08222 0.06826 0.07658]  mean=+0.07416  wins(A/tie/B)=5/0/0
  Direct XGBoost vs Tuned two-stage        [ 0.00155 -0.00344  0.00728 -0.00664  0.0042 ]  mean=+0.00059  wins(A/tie/B)=3/0/2
  Direct XGBoost vs Direct LightGBM        [0.00133 0.00174 0.00174 0.00085 0.00095]  mean=+0.00132  wins(A/tie/B)=5/0/0


,contrast,mean_diff,median_diff,sd_diff,se_diff,winsA,ties,winsB
0,Tuned DL ensemble vs Direct XGBoost,0.07357,0.07490,0.00256,0.00114,5,0,0
1,Tuned DL ensemble vs Tuned two-stage,0.07416,0.07658,0.00671,0.00300,5,0,0
2,Direct XGBoost vs Tuned two-stage,0.00059,0.00155,0.00564,0.00252,3,0,2
3,Direct XGBoost vs Direct LightGBM,0.00132,0.00133,0.00043,0.00019,5,0,0


## 5 — Paired *t*-test

Two-sided `scipy.stats.ttest_rel` per contrast. The degrees of freedom are $n - 1 = 4$. We also
run a light sanity scan of the five paired differences for any obvious extreme value.

> With only five observations, normality **cannot** be established; the *t*-test is used as a
> conventional primary procedure, not because the differences are known to be Gaussian.

In [6]:
for r in results:
    t_stat, p_raw = stats.ttest_rel(paired[r["colA"]], paired[r["colB"]])
    r["t_stat"] = float(t_stat)
    r["df"] = N_RUNS - 1
    r["p_ttest_raw"] = float(p_raw)

assert all(r["df"] == 4 for r in results), "degrees of freedom should be 4 (= n - 1)"

# Sanity scan for extreme paired differences (flag any point far from the others).
for r in results:
    d = r["diffs"]
    if r["sd_diff"] > 0:
        z = (d - d.mean()) / r["sd_diff"]
        if np.any(np.abs(z) > 2.5):
            print(f"[check] {r['contrast']}: unusually large paired diff, z-scores = "
                  f"{np.round(z, 2)}")

print("Paired t-tests (two-sided, df = 4). Normality is NOT claimed at n = 5.")
ttest_tbl = pd.DataFrame([{"contrast": r["contrast"], "t_stat": r["t_stat"],
                          "df": r["df"], "p_ttest_raw": r["p_ttest_raw"]} for r in results])
ttest_tbl.round(4)

Paired t-tests (two-sided, df = 4). Normality is NOT claimed at n = 5.


,contrast,t_stat,df,p_ttest_raw
0,Tuned DL ensemble vs Direct XGBoost,64.3556,4,0.0000
1,Tuned DL ensemble vs Tuned two-stage,24.7187,4,0.0000
2,Direct XGBoost vs Tuned two-stage,0.2339,4,0.8266
3,Direct XGBoost vs Direct LightGBM,6.9482,4,0.0023


## 6 — Exact paired permutation (sign-flip) sensitivity test

An exact two-sided sign-flip permutation test enumerates all $2^5 = 32$ sign assignments to the
five paired differences, using $|\text{mean paired difference}|$ as the statistic.

> **Resolution caveat.** With five paired runs the observed assignment plus its mirror always
> satisfy the two-sided condition, so the smallest attainable p-value is $2/32 = 0.0625$. The
> exact permutation test therefore **cannot** produce a p-value below 0.05 here — it is a
> robustness check, not the primary inferential test.

In [7]:
def exact_sign_flip_p(d):
    """Exact two-sided sign-flip permutation p-value; statistic = |mean(d)|."""
    d = np.asarray(d, dtype=float)
    n = len(d)
    obs = abs(np.mean(d))
    total = 0
    count = 0
    for signs in itertools.product((1.0, -1.0), repeat=n):
        stat = abs(np.mean(np.asarray(signs) * d))
        total += 1
        if stat >= obs - 1e-15:   # >= observed, with tiny numerical slack
            count += 1
    return count / total, total


for r in results:
    p_perm, n_perm = exact_sign_flip_p(r["diffs"])
    r["p_perm"] = float(p_perm)
    r["n_perm"] = int(n_perm)

print("Exact sign-flip permutation test: 2^5 = 32 assignments; statistic = |mean paired diff|.")
print("Smallest attainable two-sided p at n = 5 is 2/32 = 0.0625 (coarse; robustness check only).")
perm_tbl = pd.DataFrame([{"contrast": r["contrast"], "p_perm": r["p_perm"],
                         "n_perms": r["n_perm"]} for r in results])
perm_tbl.round(4)

Exact sign-flip permutation test: 2^5 = 32 assignments; statistic = |mean paired diff|.
Smallest attainable two-sided p at n = 5 is 2/32 = 0.0625 (coarse; robustness check only).


,contrast,p_perm,n_perms
0,Tuned DL ensemble vs Direct XGBoost,0.0625,32
1,Tuned DL ensemble vs Tuned two-stage,0.0625,32
2,Direct XGBoost vs Tuned two-stage,0.8125,32
3,Direct XGBoost vs Direct LightGBM,0.0625,32


## 7 — Wilcoxon signed-rank sensitivity check

Two-sided `scipy.stats.wilcoxon` on the paired differences, with zero differences handled
explicitly. This is a **sensitivity analysis only**: the Wilcoxon test has very low power at
$n = 5$ and should not be over-interpreted.

In [8]:
def paired_wilcoxon(d):
    """Two-sided exact Wilcoxon signed-rank; robust to scipy's mode/method arg rename."""
    d = np.asarray(d, dtype=float)
    n_zero = int(np.sum(d == 0))
    if np.all(d == 0):
        return np.nan, 1.0, n_zero
    kw = dict(alternative="two-sided", zero_method="wilcox")
    try:
        w, p = stats.wilcoxon(d, method="exact", **kw)
    except TypeError:
        try:
            w, p = stats.wilcoxon(d, mode="exact", **kw)
        except Exception:
            w, p = stats.wilcoxon(d, **kw)
    except ValueError:
        w, p = stats.wilcoxon(d, **kw)
    return float(w), float(p), n_zero


for r in results:
    try:
        w, p_w, n_zero = paired_wilcoxon(r["diffs"])
    except Exception as e:
        w, p_w, n_zero = np.nan, np.nan, int(np.sum(r["diffs"] == 0))
        print(f"[warn] Wilcoxon failed for {r['contrast']}: {e}")
    r["wilcoxon_stat"] = w
    r["p_wilcoxon"] = p_w
    r["n_zero_diff"] = n_zero

print("Wilcoxon signed-rank (two-sided) — SENSITIVITY only; power is very low at n = 5.")
wilcox_tbl = pd.DataFrame([{"contrast": r["contrast"], "wilcoxon_stat": r["wilcoxon_stat"],
                           "p_wilcoxon": r["p_wilcoxon"], "n_zero_diff": r["n_zero_diff"]}
                          for r in results])
wilcox_tbl.round(4)

Wilcoxon signed-rank (two-sided) — SENSITIVITY only; power is very low at n = 5.


,contrast,wilcoxon_stat,p_wilcoxon,n_zero_diff
0,Tuned DL ensemble vs Direct XGBoost,0.0,0.0625,0
1,Tuned DL ensemble vs Tuned two-stage,0.0,0.0625,0
2,Direct XGBoost vs Tuned two-stage,6.0,0.8125,0
3,Direct XGBoost vs Direct LightGBM,0.0,0.0625,0


## 8 — Multiple-comparison correction (Holm)

Holm correction is applied **separately within each test family**: once to the four paired
*t*-test p-values, and once to the four permutation p-values. P-values from different families are
never pooled into the same correction.

In [9]:
p_ttest = [r["p_ttest_raw"] for r in results]
p_perm = [r["p_perm"] for r in results]

holm_t = multipletests(p_ttest, method="holm")[1]
holm_p = multipletests(p_perm, method="holm")[1]

for r, ht, hp in zip(results, holm_t, holm_p):
    r["p_ttest_holm"] = float(ht)
    r["p_perm_holm"] = float(hp)

print("Holm correction applied SEPARATELY within each test family (t-test; permutation).")
holm_tbl = pd.DataFrame([{"contrast": r["contrast"],
                         "p_ttest_raw": r["p_ttest_raw"], "p_ttest_holm": r["p_ttest_holm"],
                         "p_perm_raw": r["p_perm"], "p_perm_holm": r["p_perm_holm"]}
                        for r in results])
holm_tbl.round(4)

Holm correction applied SEPARATELY within each test family (t-test; permutation).


,contrast,p_ttest_raw,p_ttest_holm,p_perm_raw,p_perm_holm
0,Tuned DL ensemble vs Direct XGBoost,0.0000,0.0000,0.0625,0.2500
1,Tuned DL ensemble vs Tuned two-stage,0.0000,0.0000,0.0625,0.2500
2,Direct XGBoost vs Tuned two-stage,0.8266,0.8266,0.8125,0.8125
3,Direct XGBoost vs Direct LightGBM,0.0023,0.0045,0.0625,0.2500


## 9 — Confidence intervals

The primary interval for each contrast is a **95% paired *t* confidence interval** for the mean
Macro-F1 difference:

$$\bar{d} \pm t_{0.975,\,4}\cdot \mathrm{SE}_{d}, \qquad \mathrm{SE}_d = \frac{s_d}{\sqrt{5}}.$$

A percentile **paired bootstrap** CI (fixed seed) is added purely as a sensitivity analysis and
does **not** replace the paired *t* interval.

In [10]:
t_crit = float(stats.t.ppf(0.975, df=N_RUNS - 1))
print(f"t_(0.975, 4) = {t_crit:.4f}")

BOOT_N = 10000
rng = np.random.default_rng(RNG_SEED)

for r in results:
    # Primary: paired t 95% CI.
    r["ci_lo"] = r["mean_diff"] - t_crit * r["se_diff"]
    r["ci_hi"] = r["mean_diff"] + t_crit * r["se_diff"]
    # Sensitivity: percentile paired bootstrap over the 5 paired differences.
    d = r["diffs"]
    boot_means = rng.choice(d, size=(BOOT_N, len(d)), replace=True).mean(axis=1)
    r["boot_ci_lo"] = float(np.percentile(boot_means, 2.5))
    r["boot_ci_hi"] = float(np.percentile(boot_means, 97.5))

print("Primary = paired t 95% CI. Bootstrap CI is a sensitivity check only.")
ci_tbl = pd.DataFrame([{"contrast": r["contrast"], "mean_diff": r["mean_diff"],
                       "t_ci_lo": r["ci_lo"], "t_ci_hi": r["ci_hi"],
                       "boot_ci_lo": r["boot_ci_lo"], "boot_ci_hi": r["boot_ci_hi"]}
                      for r in results])
ci_tbl.round(4)

t_(0.975, 4) = 2.7764
Primary = paired t 95% CI. Bootstrap CI is a sensitivity check only.


,contrast,mean_diff,t_ci_lo,t_ci_hi,boot_ci_lo,boot_ci_hi
0,Tuned DL ensemble vs Direct XGBoost,0.0736,0.0704,0.0767,0.0713,0.0753
1,Tuned DL ensemble vs Tuned two-stage,0.0742,0.0658,0.0825,0.0689,0.0792
2,Direct XGBoost vs Tuned two-stage,0.0006,-0.0064,0.0076,-0.0038,0.0049
3,Direct XGBoost vs Direct LightGBM,0.0013,0.0008,0.0019,0.0010,0.0017


## 10 — Final statistical-results table

One clean table with the mean Macro-F1 difference, 95% paired *t* CI, wins-out-of-5, paired *t*
statistic, raw and Holm-adjusted *t* p-values, exact and Holm-adjusted permutation p-values, and
the Wilcoxon p-value. Saved at full precision to
`outputs/statistics/headline_paired_model_tests.csv`; displayed with sensible rounding.

In [11]:
final_rows = []
for r in results:
    final_rows.append({
        "Contrast": r["contrast"],
        "MeanMacroF1Diff": r["mean_diff"],
        "CI95_paired_t": f"[{r['ci_lo']:.4f}, {r['ci_hi']:.4f}]",
        "WinsOutOf5": f"{r['winsA']}/{N_RUNS}",
        "paired_t_stat": r["t_stat"],
        "p_ttest_raw": r["p_ttest_raw"],
        "p_ttest_holm": r["p_ttest_holm"],
        "p_perm_exact": r["p_perm"],
        "p_perm_holm": r["p_perm_holm"],
        "p_wilcoxon": r["p_wilcoxon"],
    })
final = pd.DataFrame(final_rows)

out_final = OUT_DIR / "headline_paired_model_tests.csv"
final.to_csv(out_final, index=False)   # full precision retained on disk
print("Saved:", out_final)

# Rounded display only (CSV keeps full precision).
disp = final.copy()
disp["MeanMacroF1Diff"] = disp["MeanMacroF1Diff"].round(4)
disp["paired_t_stat"] = disp["paired_t_stat"].round(3)
for c in ["p_ttest_raw", "p_ttest_holm", "p_perm_exact", "p_perm_holm", "p_wilcoxon"]:
    disp[c] = disp[c].round(4)
disp

Saved: outputs/statistics/headline_paired_model_tests.csv


,Contrast,MeanMacroF1Diff,CI95_paired_t,WinsOutOf5,paired_t_stat,p_ttest_raw,p_ttest_holm,p_perm_exact,p_perm_holm,p_wilcoxon
0,Tuned DL ensemble vs Direct XGBoost,0.0736,"[0.0704, 0.0767]",5/5,64.356,0.0000,0.0000,0.0625,0.2500,0.0625
1,Tuned DL ensemble vs Tuned two-stage,0.0742,"[0.0658, 0.0825]",5/5,24.719,0.0000,0.0000,0.0625,0.2500,0.0625
2,Direct XGBoost vs Tuned two-stage,0.0006,"[-0.0064, 0.0076]",3/5,0.234,0.8266,0.8266,0.8125,0.8125,0.8125
3,Direct XGBoost vs Direct LightGBM,0.0013,"[0.0008, 0.0019]",5/5,6.948,0.0023,0.0045,0.0625,0.2500,0.0625


## 11 — Dissertation-ready text

**Block A** summarises the methods; **Block B** summarises the results per contrast. The word
"significant" is only used where the Holm-adjusted paired *t* p-value supports it, and statistical
evidence is kept distinct from the practical size of the Macro-F1 difference.

In [12]:
ALPHA = 0.05

# --- Block A: methods ---
methods = "\n".join([
    "METHODS",
    f"- The five matched user-grouped repeated-holdout runs (seeds {RUN_SEEDS}) were treated as "
    "paired observations.",
    "- The primary inferential procedure was a two-sided paired t-test on run-level Macro-F1 "
    "differences (n = 5, df = 4).",
    "- Holm correction was applied across the four pre-specified contrasts (within each test "
    "family).",
    "- An exact sign-flip permutation test (32 assignments) and a Wilcoxon signed-rank test were "
    "used as sensitivity analyses.",
    "- With only five repeated runs, inferential power and assumption checking are limited; "
    "normality is not claimed.",
])
print(methods)
print()

# --- Block B: results ---
def evidence_phrase(r):
    if r["p_ttest_holm"] < ALPHA:
        return "statistically significant after Holm adjustment"
    if r["p_ttest_raw"] < ALPHA:
        return "nominally significant (raw) but NOT after Holm adjustment"
    return "not statistically significant"

lines = ["RESULTS (statistical evidence is distinct from the practical size of the difference)"]
for r in results:
    lines.append(
        f"- {r['contrast']}: mean Macro-F1 difference {r['mean_diff']:+.4f} "
        f"(95% paired t CI [{r['ci_lo']:.4f}, {r['ci_hi']:.4f}]); "
        f"t(4) = {r['t_stat']:.3f}, raw p = {r['p_ttest_raw']:.4f}, "
        f"Holm p = {r['p_ttest_holm']:.4f}; won {r['winsA']} of {N_RUNS} runs. "
        f"-> {evidence_phrase(r)}.")
print("\n".join(lines))

METHODS
- The five matched user-grouped repeated-holdout runs (seeds [42, 43, 44, 45, 46]) were treated as paired observations.
- The primary inferential procedure was a two-sided paired t-test on run-level Macro-F1 differences (n = 5, df = 4).
- Holm correction was applied across the four pre-specified contrasts (within each test family).
- An exact sign-flip permutation test (32 assignments) and a Wilcoxon signed-rank test were used as sensitivity analyses.
- With only five repeated runs, inferential power and assumption checking are limited; normality is not claimed.

RESULTS (statistical evidence is distinct from the practical size of the difference)
- Tuned DL ensemble vs Direct XGBoost: mean Macro-F1 difference +0.0736 (95% paired t CI [0.0704, 0.0767]); t(4) = 64.356, raw p = 0.0000, Holm p = 0.0000; won 5 of 5 runs. -> statistically significant after Holm adjustment.
- Tuned DL ensemble vs Tuned two-stage: mean Macro-F1 difference +0.0742 (95% paired t CI [0.0658, 0.0825]); t(4

## 12 — Compact LaTeX table (booktabs)

Export a booktabs LaTeX version of the final results table to
`outputs/statistics/headline_paired_model_tests.tex`. Requires `\usepackage{booktabs}` in the
document preamble. Decimal places are kept modest.

In [13]:
def latex_escape(s):
    """Escape LaTeX special characters in plain text (model / contrast names)."""
    repl = {"\\": r"\textbackslash{}", "&": r"\&", "%": r"\%", "$": r"\$",
            "#": r"\#", "_": r"\_", "{": r"\{", "}": r"\}", "~": r"\textasciitilde{}",
            "^": r"\textasciicircum{}"}
    return "".join(repl.get(ch, ch) for ch in str(s))


headers = ["Contrast", r"$\Delta$ Macro-F1", r"95\% CI", "Wins",
           "$t(4)$", "$p_{t}$", "$p_{t,\\mathrm{Holm}}$",
           "$p_{\\mathrm{perm}}$", "$p_{\\mathrm{Wilcox}}$"]

body_rows = []
for r in results:
    cells = [
        latex_escape(r["contrast"]),
        f"{r['mean_diff']:+.4f}",
        f"[{r['ci_lo']:.4f}, {r['ci_hi']:.4f}]",
        f"{r['winsA']}/{N_RUNS}",
        f"{r['t_stat']:.3f}",
        f"{r['p_ttest_raw']:.4f}",
        f"{r['p_ttest_holm']:.4f}",
        f"{r['p_perm']:.4f}",
        f"{r['p_wilcoxon']:.4f}",
    ]
    body_rows.append(" & ".join(cells) + r" \\")

latex = "\n".join([
    r"% Requires \usepackage{booktabs} in the preamble.",
    r"\begin{table}[t]",
    r"\centering",
    r"\caption{Paired comparisons of headline models on run-level Macro-F1 "
    r"(five matched user-grouped holdout runs, $n=5$). CIs are paired $t$ 95\% intervals; "
    r"permutation and Wilcoxon columns are sensitivity analyses.}",
    r"\label{tab:headline_paired_tests}",
    r"\begin{tabular}{lcccccccc}",
    r"\toprule",
    " & ".join(headers) + r" \\",
    r"\midrule",
    *body_rows,
    r"\bottomrule",
    r"\end{tabular}",
    r"\end{table}",
])

out_tex = OUT_DIR / "headline_paired_model_tests.tex"
out_tex.write_text(latex)
print(latex)
print("\nSaved:", out_tex)

% Requires \usepackage{booktabs} in the preamble.
\begin{table}[t]
\centering
\caption{Paired comparisons of headline models on run-level Macro-F1 (five matched user-grouped holdout runs, $n=5$). CIs are paired $t$ 95\% intervals; permutation and Wilcoxon columns are sensitivity analyses.}
\label{tab:headline_paired_tests}
\begin{tabular}{lcccccccc}
\toprule
Contrast & $\Delta$ Macro-F1 & 95\% CI & Wins & $t(4)$ & $p_{t}$ & $p_{t,\mathrm{Holm}}$ & $p_{\mathrm{perm}}$ & $p_{\mathrm{Wilcox}}$ \\
\midrule
Tuned DL ensemble vs Direct XGBoost & +0.0736 & [0.0704, 0.0767] & 5/5 & 64.356 & 0.0000 & 0.0000 & 0.0625 & 0.0625 \\
Tuned DL ensemble vs Tuned two-stage & +0.0742 & [0.0658, 0.0825] & 5/5 & 24.719 & 0.0000 & 0.0000 & 0.0625 & 0.0625 \\
Direct XGBoost vs Tuned two-stage & +0.0006 & [-0.0064, 0.0076] & 3/5 & 0.234 & 0.8266 & 0.8266 & 0.8125 & 0.8125 \\
Direct XGBoost vs Direct LightGBM & +0.0013 & [0.0008, 0.0019] & 5/5 & 6.948 & 0.0023 & 0.0045 & 0.0625 & 0.0625 \\
\bottomrule
\end{tab

## 13 — Validation and audit

Final audit block: every file loaded, the run seeds used, the paired run-level table, whether the
published summary values were reproduced, and every output path written.

In [14]:
print("=" * 74)
print("VALIDATION & AUDIT")
print("=" * 74)

loaded_files = sorted({MODEL_SPECS[k]["runs_file"] for k in MODEL_SPECS}
                      | {MODEL_SPECS[k]["summary_file"] for k in MODEL_SPECS})
print("\nInput files loaded:")
for f in loaded_files:
    p = DATA_DIR / f
    print(f"   {p}  {'OK' if p.exists() else 'MISSING'}")

print("\nRun seeds used:", sorted(RUN_SEEDS), f"(n = {N_RUNS})")

print("\nPaired run-level table:")
print(paired.to_string(index=False))

print(f"\nPublished summary values reproduced within tol {TOL}: {ALL_REPRODUCED}")
print(summary_check[["model", "mean", "published_mean", "se", "published_se", "reproduced"]]
      .round(6).to_string(index=False))

print("\nOutput files written:")
for p in [out_paired, out_summary, out_final, out_tex]:
    print(f"   {Path(p).resolve()}  {'OK' if Path(p).exists() else 'MISSING'}")

print("\nDone.")

VALIDATION & AUDIT

Input files loaded:
   KKBoxData/clf_02b_runs.csv  OK
   KKBoxData/clf_02b_summary.csv  OK
   KKBoxData/clf_direct_holdout_runs.csv  OK
   KKBoxData/clf_direct_holdout_summary.csv  OK
   KKBoxData/clf_dl_holdout_runs.csv  OK
   KKBoxData/clf_dl_holdout_summary.csv  OK
   KKBoxData/clf_twostage_runs.csv  OK
   KKBoxData/clf_twostage_summary.csv  OK

Run seeds used: [42, 43, 44, 45, 46] (n = 5)

Paired run-level table:
 seed  dl_ensemble_macro_f1  direct_xgb_macro_f1  direct_lgbm_macro_f1  twostage_tuned_macro_f1  twostage_argmax_macro_f1  twostage_ratio_macro_f1  twostage_volume_macro_f1
   42              0.512229             0.436252              0.434925                 0.434702                  0.414809                 0.345408                  0.343534
   43              0.510310             0.440650              0.438905                 0.444085                  0.416022                 0.354118                  0.351871
   44              0.516534             